# 残差网络（ResNet）

## 残差块

In [1]:
# 导包
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

class Residual(nn.Module):
    # 参1：输入通道数；参2：输出通道数；参3：是否使用1x1卷积层进行高宽变化；参4：步长
    def __init__(self, input_channels, num_channels, use_1x1conv = False, strides = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size = 3, padding = 1, stride = strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size = 3, padding = 1)        # 高宽不变
        if use_1x1conv:         # 是否使用1x1卷积层进行高宽变化
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size = 1, stride = strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU(inplace = True)         # 是否开辟新内存：使

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:              # 判断conv3是否存在，即是否使用1x1卷积层
            X =self.conv3(X)        # ReLU是作用于Y上的，所以这里对输出X直接作用不需要ReLU
        Y += X
        return F.relu(Y)            # 合并后再做一次ReLU激活函数

## 输入和输出形状一致

In [2]:
blk = Residual(3, 3)        # 定义网络实体
X = torch.rand(4, 3, 6, 6)      # 生成随机输入张量
Y = blk(X)          # 前向传播
Y.shape

torch.Size([4, 3, 6, 6])

## 增加输出通道数的同时，减半输出的高和宽

In [3]:
blk = Residual(3, 6, use_1x1conv = True, strides = 2)       # 高宽计算：Y:(6 - 3 + 2) / 2 = 3 -> 3 - 1 + 1 = 3;X:(6 - 1) / 2 + 1 = 3
blk(X).shape

torch.Size([4, 6, 3, 3])

## ResNet模型

In [8]:
b1 = nn.Sequential(         # 输入干：网络最前面、进入残差块之前的那一小段普通卷积结构
    nn.Conv2d(1, 64, kernel_size = 7, stride = 2, padding = 1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
)

# 参1：输入通道数；参2：输出通道数；参3：残差块个数；参4：是否为第一个残差块
def resnet_block(input_channels, num_channels, num_residuals, first_block = False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:      # 如果是第一个残差块，就高宽减半
            blk.append(Residual(input_channels, num_channels, use_1x1conv = True, strides = 2))
        else:
            blk.append(Residual(num_channels, num_channels))        # 剩下块高宽不变
    return blk

b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block = True))        # ResNet阶段1，由于是第一个残差块，于是高宽减半
b3 = nn.Sequential(*resnet_block(64, 128, 2))                           # ResNet阶段2
b4 = nn.Sequential(*resnet_block(128, 256, 2))                          # ResNet阶段3
b5 = nn.Sequential(*resnet_block(256, 512, 2))                          # ResNet阶段4

net = nn.Sequential(b1, b2, b3, b4, b5, nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(512, 10))

## 观察一下ResNet中不同模块的输入形状是如何变化的

In [9]:
X = torch.rand(size = (1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 55, 55])
Sequential output shape:	 torch.Size([1, 64, 55, 55])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 10])


## 训练模型

In [ ]:
lr, num_epochs, batch_size = 0.05, 10, 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize = 224)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())